# FRED 4-Channel Drone Detection
**Physics-driven multi-channel event camera input for improved drone detection**

Run cells top to bottom in order.

**Before starting:** Runtime → Change runtime type → T4 GPU

## Cell 1 — Install & Mount Drive

In [ ]:
# Install ultralytics (everything else is pre-installed in Colab)
!pip install ultralytics -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('PyTorch      :', torch.__version__)

ModuleNotFoundError: No module named 'google'

## Cell 2 — Copy Project Files to Colab

In [ ]:
import shutil, os, sys

# Source: your project folder on Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/ai_drone/4channel_project'

# Destination: fast local SSD in Colab (much faster than Drive)
LOCAL_PROJECT = '/content/4channel_project'

# Copy project code to local SSD
if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)

# Change working directory
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

## Cell 3 — Verify Paths & Config

In [ ]:
from config import *

print('Environment : COLAB' if IN_COLAB else 'Environment : LOCAL')
print('Sequence dir:', SEQUENCE_DIR)
print('Raw file    :', RAW_FILE)
print('Coords file :', COORDS_FILE)
print('Dataset dir :', DATASET_DIR)
print()

# Check files exist
import os
for path, name in [(RAW_FILE, 'events.raw'), (COORDS_FILE, 'coordinates.txt')]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1e6 if exists else 0
    status = f'OK ({size:.1f} MB)' if exists else 'NOT FOUND'
    print(f'  {name:20s}: {status}')

if not os.path.exists(RAW_FILE):
    print()
    print('ERROR: Make sure your FRED folder 7 is uploaded to:')
    print('  Google Drive → MyDrive → ai_drone → data_from_fred → 7')

## Cell 4 — Test EVT3 Reader

In [ ]:
from evt3_reader import EVT3Reader

reader = EVT3Reader(RAW_FILE)

print('\nHeader info:')
for k, v in reader.read_header().items():
    print(f'  {k}: {v}')

# Read first window where drone is present (annotations start at ~9.87s)
T_START = 9_870_000   # 9.87s in microseconds
T_END   = T_START + WINDOW_US

print(f'\nReading window t={T_START/1e6:.3f}s – {T_END/1e6:.3f}s...')
events = reader.read_window(T_START, T_END)
print(f'  Raw events    : {len(events):,}')
print(f'  Time range    : {events["t"].min()} – {events["t"].max()} us')
print(f'  Positive evts : {(events["p"]==1).sum():,}')
print(f'  Negative evts : {(events["p"]==0).sum():,}')

## Cell 5 — Test Filters

In [ ]:
from filters import apply_filters

print('Applying noise filters...')
events_clean = apply_filters(events)
print(f'\nResult: {len(events):,} → {len(events_clean):,} events')
print(f'Noise removed: {100*(len(events)-len(events_clean))/len(events):.1f}%')

## Cell 6 — Test 4-Channel Generation & Preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from channels import generate_channels

channels = generate_channels(events_clean, T_START, T_END)
print(f'Output shape: {channels.shape}  (4 channels, {IMG_H}x{IMG_W})')

# Preview all 4 channels
titles = [
    'Ch1: Positive polarity\n(leading edge — where drone is going)',
    'Ch2: Negative polarity\n(trailing edge — where drone came from)',
    'Ch3: Rotor frequency map\n(spinning motor signature)',
    'Ch4: Time surface\n(most recent activity per pixel)',
]

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle(f'4-Channel Input  t={T_START/1e6:.3f}s', fontsize=14)

for i, (ax, title) in enumerate(zip(axes.flat, titles)):
    ch = channels[i]
    ax.imshow(ch, cmap='hot', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    n_active = (ch > 0).sum()
    ax.set_xlabel(f'Active pixels: {n_active:,}')

plt.tight_layout()
plt.savefig('/content/channel_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Preview saved to /content/channel_preview.png')

## Cell 7 — Build Training Dataset

In [ ]:
from dataset_builder import build_dataset, print_dataset_stats

print('Building dataset...')
print('This takes ~20-30 minutes for the full folder 7.')
print('Dataset saves to local Colab SSD (fast).')
print()

build_dataset()

print('\nDataset statistics:')
print_dataset_stats()

## Cell 8 — Train YOLO (4 channels)

In [ ]:
from train_4ch_yolo import train_with_ultralytics

results = train_with_ultralytics()

print('\nTraining complete!')
print(f'mAP50     : {results.results_dict.get("metrics/mAP50(B)", 0)*100:.2f}%')
print(f'Paper base: 87.68%')

## Cell 9 — Evaluate & Compare vs Paper

In [ ]:
from evaluate import evaluate
evaluate()

## Cell 10 — Save Results Back to Drive

In [ ]:
import shutil, os

# Save trained model back to Google Drive
src_model = f'/content/runs/fred_4channel/weights/best.pt'
dst_model = '/content/drive/MyDrive/ai_drone/4channel_project/best_model.pt'

if os.path.exists(src_model):
    shutil.copy(src_model, dst_model)
    print(f'Model saved to Drive: {dst_model}')
else:
    print('No model found — run Cell 8 first')

# Save channel preview
shutil.copy(
    '/content/channel_preview.png',
    '/content/drive/MyDrive/ai_drone/4channel_project/channel_preview.png'
)
print('Preview saved to Drive')